In [1]:
%run -i ../../python_scripts/nb_setup.py

c:\Users\ejeme\Documents\python_repos\selective-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### chest X-ray pathology detection

**Task.** Binary detection of any pathology on NIH ChestX-ray14, with a
DenseNet-121 from `torchxrayvision` **trained on CheXpert (not by us)** — so ChestX-ray14 is
data the classifier never saw. No training here, inference only.

**Output.** A pickled `DataFrame` with `y_true`, `y_pred`, `kappa` (softmax
response, in `(0.5, 1)`), directly usable with `theta_min, theta_max = 0.5, 1`.

**Requirements.**
```
pip install torchxrayvision
```
NIH images (`images_001.tar.gz` ... `images_012.tar.gz`) from
<https://nihcc.app.box.com/v/ChestXray-NIHCC>.
The label CSV is already inside `torchxrayvision`.

In [2]:
IMGPATH = "D:/CXR8/CXR8/images"  # flat folder of the NIH .png files
WEIGHTS = "densenet121-res224-chex"  # CheXpert-trained -> NIH is unseen
PATHOLOGY = (
    "Pneumothorax"  # 4.7% prevalence; "Effusion" (12%) for a larger positive stratum
)
N_MAX = 30_000  # cap on images pushed through the net
P_TAU = 0.2  # share of samples reserved to fit the threshold
SEED, OUT = 0, "sgp_set_densenet_SR"

rng = np.random.default_rng(SEED)
torch.set_num_threads(os.cpu_count())

In [ ]:
model = xrv.models.DenseNet(weights=WEIGHTS).eval()
tf = torchvision.transforms.Compose(
    [xrv.datasets.XRayCenterCrop(), xrv.datasets.XRayResizer(224)]
)
d = xrv.datasets.NIH_Dataset(
    imgpath=IMGPATH, transform=tf, views=["PA", "AP"], unique_patients=False
)
xrv.datasets.relabel_dataset(model.pathologies, d)

# One random film per patient, keeping the i.i.d.
csv = d.csv.reset_index(drop=True)
idx = (
    csv.sample(frac=1, random_state=SEED).drop_duplicates("Patient ID").index.to_numpy()
)
prev = pd.Series(np.nanmean(d.labels[idx], axis=0), index=model.pathologies).dropna()
print(len(d), "images ->", len(idx), "patients")
print(
    (prev[prev > 0] * len(idx))
    .astype(int)
    .rename("positives")
    .to_frame()
    .assign(prevalence=prev[prev > 0].round(4))
    .sort_values("positives", ascending=False)
)

If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/chex-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O C:\Users\ejeme\.torchxrayvision\models_data/chex-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
{'Fibrosis', 'Emphysema', 'Infiltration', 'Nodule', 'Hernia', 'Mass', 'Pleural_Thickening'} will be dropped
 doesn't exist. Adding nans instead.
 doesn't exist. Adding nans instead.
 doesn't exist. Adding nans instead.
 doesn't exist. Adding nans instead.
 doesn't exist. Adding nans instead.
 doesn't exist. Adding nans instead.
 doesn't exist. Adding nans instead.
Lung Lesion doesn't exist. Adding nans instead.
Fracture doesn't exist. Adding nans instead.
Lung Opacity doesn't exist. Adding nans instead.
Enlarged Cardiomediastinum doesn't exist. Adding nans instead.
30805 images -> 30000 used; Pneumothorax prevalence 0.0083


In [ ]:
assert PATHOLOGY in model.pathologies, model.pathologies
col = model.pathologies.index(PATHOLOGY)
sel = rng.permutation(idx)[:N_MAX]
ds = xrv.datasets.SubsetDataset(d, sel)
n_pos = int(np.nansum(d.labels[sel, col]))
print(f"{PATHOLOGY}: N={len(ds)} | positives={n_pos} ({n_pos/len(ds):.4f})")
assert n_pos > 500, "positive stratum too thin for a useful FNR/SE bound"

In [ ]:
# Forward pass
loader = torch.utils.data.DataLoader(ds, batch_size=32, num_workers=4)
p, y = [], []
with torch.no_grad():
    for b in tqdm(loader):
        p.append(model(b["img"]).numpy()[:, col])
        y.append(b["lab"].numpy()[:, col])
p, y = np.concatenate(p), np.concatenate(y)

m = ~np.isnan(y)
p, y = p[m], y[m].astype(int)
print("N =", len(y), "| AUC =", round(roc_auc_score(y, p), 3))

In [ ]:
# Threshold fitted on a disjoint slice (Youden's J), then absorbed into the head's bias
tau_split = rng.random(len(y)) < P_TAU
fpr, tpr, thr = roc_curve(y[tau_split], p[tau_split])
tau = float(np.clip(thr[np.argmax(tpr - fpr)], 1e-6, 1 - 1e-6))

s = logit(np.clip(p, 1e-6, 1 - 1e-6)) - logit(
    tau
)  # decision margin of the shifted head
p_shift = expit(s)
sgp_df = pd.DataFrame(
    {
        "y_true": y.astype(float),
        "y_pred": (s >= 0).astype(float),
        "kappa": np.maximum(p_shift, 1 - p_shift),  # softmax response
    }
)[~tau_split].reset_index(drop=True)
print("tau =", round(tau, 4))

In [ ]:
def report(df, name):
    n = len(df)
    err = (df.y_pred != df.y_true).mean()
    fp = ((df.y_pred == 1) & (df.y_true == 0)).sum()
    fn = ((df.y_pred == 0) & (df.y_true == 1)).sum()
    print(
        f"{name}: N={n} | positives={df.y_true.mean():.3f} | 0/1 risk={err:.3f} | "
        f"FP={fp} FN={fn} | FPR={fp/(df.y_true==0).sum():.3f} "
        f"FNR={fn/(df.y_true==1).sum():.3f} | "
        f"kappa in [{df.kappa.min():.3f}, {df.kappa.max():.3f}]"
    )


report(sgp_df, "sgp_set")
pickle.dump(sgp_df, open(OUT, "wb"))
sgp_df.head(3)